# Практическая работа № 1. Классификация
## Активность человека по мобильным сенсорам
**Дисциплина:** Прикладной искусственный интеллект.

**Выполнил(а):** ФИО, ИТМО ID.

Авторы практикума: Ким Станислав Александрович, Евстафьев Олег Александрович.

Исследуйте несколько моделей и гиперпараметров, объясните precision, recall и F1, визуализируйте результаты. Файлы этой работы относятся к активности человека, а не к жилью.

**Первое знакомство с ML.** Одна строка описывает короткий фрагмент активности человека. Модель получает числовые признаки и предсказывает вид активности. Сначала получите исходный результат, затем меняйте настройки.

`fit` означает обучение, `predict` — получение прогноза. Валидация нужна для выбора настроек, внешний тест — для итоговой проверки. Правильный ответ и номер участника не должны попасть в признаки.

Выполняйте ячейки сверху вниз. После изменения исходных данных или перед сдачей перезапустите среду и выполните всё заново. Для запуска скачайте весь проект: учебные модули находятся в общей папке `code`, а данные — в папке `data` рядом с этим ноутбуком.

## 1. Среда и данные
Файлы `train.csv` и `test.csv` уже находятся в папке `data` этого проекта. Запускайте ноутбук в Jupyter из своей папки работы или из корня скачанного проекта. Колонки `Activity` и `subject` исключаются по именам. Если идентификатор участника называется иначе, укажите реальное имя в `group_column`.

In [ ]:
from pathlib import Path
import sys
import os
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (p / "code").is_dir() and (p / "labs").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the complete practicum project with code/ and labs/")
LAB_DIR = PROJECT_ROOT / "labs" / "01_classification"
sys.path.insert(0, str(PROJECT_ROOT / "code"))
os.chdir(LAB_DIR)
print("Working directory:", LAB_DIR)
import pandas as pd
import matplotlib.pyplot as plt
from lab1_utils import load_har, candidate_models, fit_validation
from lab1_utils import CLASSES, final_test, save_har_result
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
display(train.head())
display(train["Activity"].value_counts())
print("Participants:", train["subject"].nunique(), test["subject"].nunique())


**Ваше объяснение:** что является объектом, признаком, меткой и группой? Разберите два признака; проверьте пропуски и словарь меток.

In [ ]:
data = load_har("data", group_column="subject", seed=40)
print(data["Xtr"].shape, data["Xva"].shape)
print(data["training_subjects"], data["validation_subjects"])
assert not set(data["training_subjects"]) & set(data["validation_subjects"])
assert "Activity" not in data["features"] and "subject" not in data["features"]


## 2. Гипотезы до запуска
Объясните выбор KNN, LinearSVC и леса. Предскажите влияние k, C и глубины. D0 — константный ориентир. Сравнение проводится на одной групповой валидации; внешний тест пока не используется.

In [ ]:
models = candidate_models(seed=40)
rows = []
for name, estimator in models.items():
    result = fit_validation(data, estimator)
    rows.append({"model": name, **result["metrics"], "seconds_fit": result["seconds_fit"]})
comparison = pd.DataFrame(rows)
display(comparison.sort_values("macro_f1", ascending=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison.plot.bar(x="model", y="macro_f1", ax=axes[0], legend=False)
comparison.plot.bar(x="model", y="seconds_fit", ax=axes[1], legend=False)
axes[0].set_ylabel("Validation macro-F1")
axes[1].set_ylabel("Fit time, seconds")
fig.tight_layout()


## 3. Зафиксируйте выбор
Опишите наблюдения по каждому семейству. При близких результатах обсудите сложность и время. Строка ниже выбирает максимальный macro-F1 валидации; при изменении правила сделайте это до просмотра теста.

In [ ]:
selected = comparison.sort_values(["macro_f1", "model"], ascending=[False, True]).iloc[0]["model"]
print("Selected before final test:", selected)


## 4. Итоговый тест
Выбранный Pipeline переобучается на всём train.csv. После просмотра теста не подбирайте новые гиперпараметры по нему.

In [ ]:
final = final_test(data, models[selected])
yhat = final["prediction"]
display(pd.DataFrame(final["report"]).T)
from sklearn.metrics import ConfusionMatrixDisplay
fig_cm, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, norm in zip(axes, [None, "true"]):
    ConfusionMatrixDisplay.from_predictions(data["ytest"], yhat, labels=range(6),
        display_labels=CLASSES, normalize=norm, xticks_rotation=90, ax=ax)
fig_cm.tight_layout()


## 5. Сохранение
Каждому новому сравнению назначьте новый каталог. Подготовьте содержательные ответы на вопросы методички, включая различия precision/recall/F1 и влияние гиперпараметров.

In [ ]:
OUTPUT_DIR = Path("runs/lab1_01")
save_har_result(data, comparison, selected, final, OUTPUT_DIR)
fig.savefig(OUTPUT_DIR / "comparison.pdf", bbox_inches="tight")
fig_cm.savefig(OUTPUT_DIR / "confusion.pdf", bbox_inches="tight")


## Вывод
Заполните: выбранная модель; аргументы по валидации; итоговый тест; две пары перепутанных действий; ограничения; следующий проверяемый шаг.

**Самопроверка:** участники не пересекаются; scaler обучен на training; все конфигурации сохранены; тест использован после выбора.